

# Tree Models — California Housing (Regression & Classification)
### OPIM 5512 — Applied Data Science · Module 1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module1/M1_TreeModels_CAHousing.ipynb)

*A clean ML refresher with **decision trees, random forests, and gradient boosting** — first as
regression, then as classification — on the **California housing** dataset. Loads from scikit-learn,
so there's nothing to upload. **Runtime → Run all.**

## Setup

In [1]:
# !pip install -q scikit-learn pandas matplotlib seaborn
%matplotlib inline
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")

## 1. Load California housing
Target `MedHouseVal` = median home value ($100,000s).

In [2]:
from sklearn.datasets import fetch_california_housing
df = fetch_california_housing(as_frame=True).frame
print(df.shape); df.head()

(20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 2. Split & scale (fit on train only)

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
y = df["MedHouseVal"]; X = df.drop(columns="MedHouseVal")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = MinMaxScaler(); X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)

## 3. Regression — DTR vs RFR vs GBR

In [4]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
for n,m in {"DecisionTree":DecisionTreeRegressor(random_state=42),
            "RandomForest":RandomForestRegressor(random_state=42),
            "GradientBoosting":GradientBoostingRegressor(random_state=42)}.items():
    m.fit(X_tr,y_tr); p=m.predict(X_te)
    print(f"{n:18s} R2={r2_score(y_te,p):.3f}  MAE={mean_absolute_error(y_te,p):.3f}  RMSE={mean_squared_error(y_te,p)**0.5:.3f}")

DecisionTree       R2=0.622  MAE=0.454  RMSE=0.703


RandomForest       R2=0.805  MAE=0.327  RMSE=0.505


GradientBoosting   R2=0.776  MAE=0.372  RMSE=0.542


## 4. Classification — is a district above the median value?
Recode the target to a **balanced** 0/1 at the median, then DTC vs RFC vs GBC.

In [5]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report
yc = (y > y.median()).astype(int)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X, yc, test_size=0.2, random_state=42)
Xc_tr = sc.fit_transform(Xc_tr); Xc_te = sc.transform(Xc_te)
clf = RandomForestClassifier(random_state=42).fit(Xc_tr, yc_tr)
print(classification_report(yc_te, clf.predict(Xc_te)))

              precision    recall  f1-score   support

           0       0.89      0.90      0.89      2077
           1       0.90      0.89      0.89      2051

    accuracy                           0.89      4128
   macro avg       0.89      0.89      0.89      4128
weighted avg       0.89      0.89      0.89      4128



## Takeaways
- Same **read → split → scale-on-train → fit → evaluate** flow for regression and classification.
- Ensembles (RF, GBM) beat a single tree; gradient boosting usually leads on this data.
- Classification = recode to a **balanced** target, then the confusion-matrix metrics (precision/recall/F1).